## Step 4: Feature Engineering

This notebook takes the file we built in Step 2 and peforms feature engineering on the dataset we will use to train models. 
- Read **`Nhanes_Cleaned.csv`** created in step 2.
- Remove feature(s) with only one recorded value.
- Split the updated dataset into training, validation, and testing datasets
- Save the split clean datasets as-is for CatBoost modeling algorithm
- Apply one-hot encoding of categorical features to the split datasets
- Save the one-hot encoded splits for XGBoost and RandomForest modeling algorithms
- Apply standardization after one-hot encoding
- Save one-hot encoded + standardized splits for SVM and Logistic modeling algorithms

#### Import Libraries

In [6]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

#### Data Path

In [7]:
BASE_DIR = "."
CLEAN_CSV       = os.path.join(BASE_DIR, "Nhanes_Cleaned.csv")          # model-ready, raw codes

CATBOOST_TRAIN  = os.path.join(BASE_DIR,"Nhanes_Cat_Train.csv")         #split datasets (no one-hot encoding, no standardization) for CatBoost
CATBOOST_VAL    = os.path.join(BASE_DIR,"Nhanes_Cat_Val.csv")
CATBOOST_TEST   = os.path.join(BASE_DIR,"Nhanes_Cat_Test.csv")

TREE_TRAIN      = os.path.join(BASE_DIR,"Nhanes_XGB_Forest_Train.csv")  #split datasets after one-hot encoding for XGBoost and RandomForest
TREE_VAL        = os.path.join(BASE_DIR,"Nhanes_XGB_Forest_Val.csv")
TREE_TEST       = os.path.join(BASE_DIR,"Nhanes_XGB_Forest_Test.csv") 

STD_TRAIN       = os.path.join(BASE_DIR,"Nhanes_SVM_Logistic_Train.csv") #split datasets after one-hot encoding and standardization for SVM and Logistic
STD_VAL         = os.path.join(BASE_DIR,"Nhanes_SVM_Logistic_Val.csv")
STD_TEST        = os.path.join(BASE_DIR,"Nhanes_SVM_Logistic_Test.csv") 

TARGET_TRAIN    = os.path.join(BASE_DIR,"Nhanes_Target_Train.csv")      #split target vectors, no preprocessing required since only 0s and 1s
TARGET_VAL      = os.path.join(BASE_DIR,"Nhanes_Target_Val.csv")
TARGET_TEST     = os.path.join(BASE_DIR,"Nhanes_Target_Test.csv")

#### Step 4.1 Read dataset

In [8]:
#load dataframe
clean_df = pd.read_csv(CLEAN_CSV,low_memory=False)
display(clean_df)

,RIAGENDR,RIDAGEYR,RIDRETH1,RIDRETH3,RIDEXMON,DMDBORN4,DMDEDUC2,DMDMARTZ,INDFMPIR,ALQ111,...,HSQ590,INDFMMPI,INDFMMPC,PAD680,SLD012,SLD013,SMQ020,SMAQUEX2,WTMEC_COMB,Depression_Flag
0,2.0,29.0,5.0,6.0,2.0,2.0,5.0,3.0,5.00,1.0,...,2.0,5.00,3.0,480.0,7.5,8.0,2.0,1.0,5018.441965,0
1,1.0,49.0,3.0,3.0,2.0,1.0,2.0,3.0,2.49,1.0,...,2.0,1.20,1.0,60.0,10.0,13.0,1.0,1.0,5328.450999,0
2,1.0,36.0,3.0,3.0,2.0,1.0,4.0,3.0,0.83,1.0,...,1.0,0.53,1.0,180.0,6.5,8.0,1.0,1.0,13639.136523,1
3,1.0,68.0,5.0,7.0,1.0,1.0,4.0,3.0,1.20,1.0,...,2.0,1.20,1.0,300.0,9.5,9.5,2.0,1.0,4800.984750,0
4,1.0,76.0,3.0,3.0,2.0,1.0,5.0,1.0,3.61,1.0,...,2.0,3.18,3.0,900.0,7.0,8.0,1.0,1.0,19579.290154,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13726,2.0,69.0,5.0,7.0,2.0,1.0,3.0,2.0,0.98,2.0,...,2.0,1.25,1.0,360.0,8.0,10.0,1.0,1.0,18110.606192,0
13727,2.0,49.0,4.0,4.0,2.0,1.0,5.0,3.0,2.49,1.0,...,1.0,2.22,2.0,480.0,7.0,7.0,1.0,1.0,24985.511139,0
13728,1.0,50.0,2.0,2.0,1.0,2.0,4.0,1.0,1.95,1.0,...,2.0,2.50,3.0,600.0,9.0,8.0,2.0,1.0,17064.436205,0
13729,1.0,40.0,2.0,2.0,1.0,1.0,4.0,2.0,3.11,2.0,...,1.0,3.26,3.0,240.0,8.0,12.0,2.0,1.0,17788.216096,0


#### Step 4.2: Identify columns with only one value and remove them.

Columns with only one unique value (or *zero-variance columns*) provide no predictive value to models and should be removed entirely.

In [9]:
#determine columns where std dev = 0 (i.e. every value in column is identical)
clean_subset = clean_df.iloc[:, 2:]
non_varying_cols = clean_subset.columns[clean_subset.std() == 0]

#display columns and whether std dev = 0
print("Non-string columns before deletion where standard dev = 0:")
print(clean_subset.std()==0)

#drop all columns where std dev = 0
clean_df.drop(columns = non_varying_cols,inplace=True)

#after drop, display columns and whether std dev = 0 (should all be false)
print("\nNon-string columns after deletion where standard dev = 0:")
print(clean_df.iloc[:, 2:].std()==0)

Non-string columns before deletion where standard dev = 0:
RIDRETH1           False
RIDRETH3           False
RIDEXMON           False
DMDBORN4           False
DMDEDUC2           False
DMDMARTZ           False
INDFMPIR           False
ALQ111             False
ALQ121             False
ALQ130             False
ALQ142             False
ALQ151             False
ALQ170             False
HSQ590             False
INDFMMPI           False
INDFMMPC           False
PAD680             False
SLD012             False
SLD013             False
SMQ020             False
SMAQUEX2            True
WTMEC_COMB         False
Depression_Flag    False
dtype: bool

Non-string columns after deletion where standard dev = 0:
RIDRETH1           False
RIDRETH3           False
RIDEXMON           False
DMDBORN4           False
DMDEDUC2           False
DMDMARTZ           False
INDFMPIR           False
ALQ111             False
ALQ121             False
ALQ130             False
ALQ142             False
ALQ151             F

#### Step 4.3: Split the dataset into training, validation, and test sets


In [10]:
X = clean_df.drop(columns=['Depression_Flag']) # retain everything but the target Depression_Flag field
y = clean_df['Depression_Flag']

#create test sets consisting of 15% of available data
X_train_val, X_test, y_train_val, y_test = train_test_split(X,y,test_size=0.15,random_state = 4086)

#create validation set consisting of 15% of original data, and training set being what's left
X_train, X_val, y_train, y_val = train_test_split(X_train_val,y_train_val,test_size=0.15/0.85,random_state = 4086)

#### Step 4.4: Save training, validation, and test sets for Catboost and target vectors for all models as-is

Since the CatBoost algorithm natively and automatically handles text-based or discrete category features without preprocessing, we save our datasets as-is for CatBoost. Additionally, since our target vectors consist of only 0s and 1s, neither one-hot encoding nor standardization are required based on our model algorithm choices, so they are also saved as-is.

In [11]:
#write catboost split datasets to csv
X_train.to_csv(CATBOOST_TRAIN,index=False)
X_val.to_csv(CATBOOST_VAL,index=False)
X_test.to_csv(CATBOOST_TEST,index=False)

#write split target vectors to csv
y_train.to_csv(TARGET_TRAIN,index=False)
y_val.to_csv(TARGET_VAL,index=False)
y_test.to_csv(TARGET_TEST,index=False)

#### Step 4.5: Perform one-hot encoding of all categorical data in datasets

For all remaining modeling algorithms (XGBoost, Random Forest, SVM, Logistic), we will need to one-hot encode our categorical features. Since the highest category count
we have for any feature is low (under 15), one-hot encoding should be a better choice than binary encoding.

In [13]:
categorical_cols = ['RIAGENDR', 'RIDRETH1', 'RIDRETH3', 'RIDEXMON', 
                    'DMDBORN4', 'DMDEDUC2', 'DMDMARTZ', 
                    'ALQ111', 'ALQ121', 'ALQ142', 'ALQ151', 
                    'HSQ590', 'INDFMMPC', 'SMQ020']															

fitted_encoders = {}

for col in categorical_cols:

    #intitialize encoder on training data
    encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    train_encoded = encoder.fit_transform(X_train[[col]])
    #fitted_encoders[col]=encoder

    #generate one-hot encoding category names
    encoder_categories = [f"{col}_{cat}" for cat in encoder.categories_[0]]

    #transform splits using encoder
    val_encoded = encoder.transform(X_val[[col]])
    test_encoded = encoder.transform(X_test[[col]])

    #convert to dataframes
    train_labels = pd.DataFrame(train_encoded,columns=encoder_categories,index=X_train.index)
    val_labels = pd.DataFrame(val_encoded,columns=encoder_categories,index=X_val.index)
    test_labels = pd.DataFrame(test_encoded,columns=encoder_categories,index=X_test.index)

    #join one-hot encoding columns back to source dataframes
    X_train = X_train.join(train_labels)
    X_val = X_val.join(val_labels)
    X_test = X_test.join(test_labels)

    #drop original unencoded column
    X_train = X_train.drop(columns=[col])
    X_val = X_val.drop(columns=[col])
    X_test = X_test.drop(columns=[col])

#### Step 4.6: Save one-hot encoded training, validation, and test sets for XGBoost and Random Forest algorithms

Since XGBoost and Random Forest algorithms work best with encoded categories, we now save our one-hot datasets to be used for those models.

In [14]:
#write tree-friendly split datasets to csv
X_train.to_csv(TREE_TRAIN,index=False)
X_val.to_csv(TREE_VAL,index=False)
X_test.to_csv(TREE_TEST,index=False)

#### Step 4.7: Perform standardization of data in datasets for SVM and Logistic algorithms

For SVM and logistic classification, only applying one-hot encoding to our categorical features is insufficient. Since both algorithms are distance-based, standardization is necessary to minimize overrepresenting or underrepresenting features because their magnitude/units are too large or too small comparatively.

In [15]:
standard_scaler = StandardScaler()

#initialize scaler on training data 
X_train_std = standard_scaler.fit_transform(X_train)

#transform splits with scaler 
X_val_std = standard_scaler.transform(X_val)
X_test_std = standard_scaler.transform(X_test)

#build new dataframes with standardized data
X_train_std_df = pd.DataFrame(X_train_std, columns=X_train.columns, index=X_train.index)
X_val_std_df = pd.DataFrame(X_val_std, columns=X_val.columns, index=X_val.index)
X_test_std_df = pd.DataFrame(X_test_std, columns=X_test.columns, index=X_test.index)

#write standardized split datasets to csv
X_train_std_df.to_csv(STD_TRAIN,index=False)
X_val_std_df.to_csv(STD_VAL,index=False)
X_test_std_df.to_csv(STD_TEST,index=False)